# Pretrain Commutative CNN Encoder

Load the shared unlabeled pretraining dataset and save commutative CNN encoder weights for downstream classification notebooks.

In [6]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from src.ml import CommutativeCNNClassifier, CommutativeCNNConfig, LossWeightConfig, OptimizationConfig
from src.tensor_utils import load_unlabeled_tensor_dataset


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_cnn/encoder_state.pt")

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(6, 8),
    spatial_kernel_size_z=(1, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(12,),
    temporal_st_kernel_sizes=(3,),
    temporal_ts_channels=(8,),
    temporal_ts_kernel_sizes=(5,),
    spatial_agg_channels=(12,),
    spatial_agg_kernel_size_z=(3,),
    spatial_agg_kernel_size_xy=(3,),
    spatial_agg_stride_z=(1,),
    spatial_agg_stride_xy=(1,),
    spatial_agg_pool_kernel_z=(1,),
    spatial_agg_pool_kernel_xy=(2,),
    spatial_agg_pool_stride_z=(1,),
    spatial_agg_pool_stride_xy=(2,),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=8,
    num_prototypes=8,
    dropout=0.5,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=75,
    learning_rate=2e-4,
    weight_decay=3e-3,
    early_stopping_patience=8,
    early_stopping_min_delta=0.0,
    scheduler_patience=3,
    scheduler_factor=0.7,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    consistency_weight=1.0,
    feature_weight=0.05,
    prototype_temperature=0.1,
)

In [8]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
unlabeled_dataset["tensors"].shape, unlabeled_dataset["metadata"].shape

(torch.Size([400, 20, 5, 96, 96]), (400, 7))

In [9]:
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(unlabeled_dataset["tensors"])
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path

cols:
    ep=epoch
    lr=learning_rate
    eta=estimated_time_remaining
    trL=train_loss
    trCC=train_commutative_consistency_loss
    trFA=train_feature_alignment_loss
     ep       lr       eta |      trL     trCC     trFA
001/075 2.00e-04     38:20 |   4.2251   4.1994   0.5138
002/075 2.00e-04     38:26 |   3.4283   3.4110   0.3461
003/075 2.00e-04     39:13 |   2.8844   2.8711   0.2675
004/075 2.00e-04     38:55 |   2.6983   2.6883   0.2001
005/075 2.00e-04     38:40 |   2.5591   2.5508   0.1660
006/075 2.00e-04     38:26 |   2.4955   2.4882   0.1456
007/075 2.00e-04     38:14 |   2.3651   2.3587   0.1284
008/075 2.00e-04     37:47 |   2.2527   2.2467   0.1190
009/075 2.00e-04     37:25 |   2.2298   2.2245   0.1048


KeyboardInterrupt: 

In [ ]:
model.pretrain_history_.tail()

,epoch,train_loss,train_commutative_consistency_loss,train_feature_alignment_loss
58,59,2.095045,2.092163,0.057625
59,60,2.091553,2.088806,0.054939
60,61,2.093289,2.090387,0.058047
61,62,2.093209,2.090169,0.060808
62,63,2.090424,2.087778,0.052931
